In [1]:
import pymysql
import pandas as pd
import getpass

password = getpass.getpass("MySQL password: ")

conn = pymysql.connect(
    host="127.0.0.1",
    user="root",
    password=password,
    database="sakila"
)

In [2]:
def rentals_month(conn, month, year):
    query = """
        SELECT
            rental_id,
            rental_date,
            inventory_id,
            customer_id,
            return_date,
            staff_id,
            last_update
        FROM rental
        WHERE YEAR(rental_date) = %s
          AND MONTH(rental_date) = %s
    """
    
    cursor = conn.cursor()
    cursor.execute(query, (year, month))
    rows = cursor.fetchall()
    
    columns = [
        "rental_id",
        "rental_date",
        "inventory_id",
        "customer_id",
        "return_date",
        "staff_id",
        "last_update"
    ]
    
    return pd.DataFrame(rows, columns=columns)


In [3]:
may_rentals = rentals_month(conn, 5, 2005)
may_rentals.head()


,rental_id,rental_date,inventory_id,customer_id,return_date,staff_id,last_update
0,1,2005-05-24 22:53:30,367,130,2005-05-26 22:04:30,1,2006-02-15 21:30:53
1,2,2005-05-24 22:54:33,1525,459,2005-05-28 19:40:33,1,2006-02-15 21:30:53
2,3,2005-05-24 23:03:39,1711,408,2005-06-01 22:12:39,1,2006-02-15 21:30:53
3,4,2005-05-24 23:04:41,2452,333,2005-06-03 01:43:41,2,2006-02-15 21:30:53
4,5,2005-05-24 23:05:21,2079,222,2005-06-02 04:33:21,1,2006-02-15 21:30:53


In [4]:
def rental_count_month(rentals_df, month, year):
    col = f"rentals_{month:02d}_{year}"
    return (
        rentals_df
        .groupby("customer_id")
        .size()
        .reset_index(name=col)
    )


In [10]:
def compare_rentals(df_a, df_b):
    col_a = [c for c in df_a.columns if c.startswith("rentals_")][0]
    col_b = [c for c in df_b.columns if c.startswith("rentals_")][0]
    
    result = df_a.merge(df_b, on="customer_id", how="inner")
    result["difference"] = result[col_b] - result[col_a]
    return result


In [5]:
may_rentals = rentals_month(conn, 5, 2005)
jun_rentals = rentals_month(conn, 6, 2005)


In [7]:
may_counts = rental_count_month(may_rentals, 5, 2005)
jun_counts = rental_count_month(jun_rentals, 6, 2005)

may_counts.head()


,customer_id,rentals_05_2005
0,1,2
1,2,1
2,3,2
3,5,3
4,6,3


In [8]:
jun_counts.head()

,customer_id,rentals_06_2005
0,1,7
1,2,1
2,3,4
3,4,6
4,5,5


In [11]:
comparison = compare_rentals(may_counts, jun_counts)
comparison.head()

,customer_id,rentals_05_2005,rentals_06_2005,difference
0,1,2,7,5
1,2,1,1,0
2,3,2,4,2
3,5,3,5,2
4,6,3,4,1


In [12]:
comparison.sort_values("difference", ascending=False).head(10)

,customer_id,rentals_05_2005,rentals_06_2005,difference
386,454,1,10,9
389,457,1,9,8
178,213,1,9,8
248,295,1,9,8
23,27,1,8,7
194,234,1,8,7
218,260,1,8,7
322,380,1,8,7
481,561,2,9,7
345,406,1,7,6


In [13]:
comparison.sort_values("difference").head(10)

,customer_id,rentals_05_2005,rentals_06_2005,difference
173,207,6,1,-5
230,274,6,2,-4
210,250,5,1,-4
167,198,5,1,-4
509,596,6,2,-4
11,14,5,1,-4
136,161,6,2,-4
391,460,4,1,-3
186,222,5,2,-3
474,551,5,2,-3
